In [3]:
import numpy as np
import pandas as pd
from pathlib import Path

RAW = Path("../data/raw")
PROC = Path("../data/processed")
PROC.mkdir(parents=True, exist_ok=True)

GAS = ["H2", "CH4", "C2H6", "C2H4", "C2H2"]
LABEL = "Fail"

df = pd.read_csv(RAW / "dga_dataset.csv", sep=";", decimal=",")
df[LABEL] = df[LABEL].astype(str).str.strip().str.upper()

print(df.shape, df[LABEL].nunique())

(4150, 6) 7


In [4]:
dedup = df.drop_duplicates().reset_index(drop=True)

print(len(df), "->", len(dedup))

n_labels = dedup.groupby(GAS)[LABEL].transform("nunique")

conflicts = dedup[n_labels > 1].sort_values(GAS)

print(
    len(conflicts),
    "rows in",
    conflicts.groupby(GAS).ngroups,
    "conflicting profiles"
)

conflicts.head(20)

4150 -> 2758
78 rows in 39 conflicting profiles


,H2,CH4,C2H6,C2H4,C2H2,Fail
635,0.00,18900.0000,410.0000,540.0,330.0,T2
1089,0.00,18900.0000,410.0000,540.0,330.0,T1
362,1.00,27.0000,49.0000,4.0,1.0,T1
629,1.00,27.0000,49.0000,4.0,1.0,T2
995,4.32,193.0000,125.0000,118.0,0.0,T2
1133,4.32,193.0000,125.0000,118.0,0.0,T1
699,8.00,0.0001,0.0001,43.0,101.0,D2
886,8.00,0.0001,0.0001,43.0,101.0,D1
632,12.00,18.0000,4.0000,4.0,0.0,T2
1086,12.00,18.0000,4.0000,4.0,0.0,T1


In [5]:
pairs = conflicts.groupby(GAS)[LABEL].apply(
    lambda s: "/".join(sorted(s.unique()))
)

print(pairs.value_counts())

Fail
T1/T2    13
D1/PD    13
D1/D2     5
T2/T3     5
D2/PD     2
T1/T3     1
Name: count, dtype: int64


In [ ]:
clean = dedup[n_labels == 1].reset_index(drop=True)

print(clean.shape)
print(clean[LABEL].value_counts())

assert not clean.duplicated(subset=GAS).any()

clean.to_csv(PROC / "dga_clean.csv", index=False)

(2680, 6)
Fail
NF    707
T3    572
D1    336
D2    329
T1    311
T2    222
PD    203
Name: count, dtype: int64


In [7]:
clean.sort_values("H2", ascending=False).head(5)

,H2,CH4,C2H6,C2H4,C2H2,Fail
98,92600.00,10200.00,0.0010,0.0010,0.0010,PD
625,92600.00,10200.00,1.0000,2.0000,0.0001,PD
631,92600.00,10200.00,0.0001,0.0001,0.0001,PD
523,92600.00,10200.00,0.0000,0.0000,0.0000,PD
1321,74810.28,7139.15,664.8300,13471.2800,15150.6900,D1


In [8]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import cross_val_score, StratifiedKFold

eps = 1e-3

X_ratio = pd.DataFrame({
    "r_C2H2_C2H4": np.log10((clean.C2H2 + eps) / (clean.C2H4 + eps)),
    "r_CH4_H2": np.log10((clean.CH4 + eps) / (clean.H2 + eps)),
    "r_C2H4_C2H6": np.log10((clean.C2H4 + eps) / (clean.C2H6 + eps)),
})

X_raw = np.log10(clean[GAS] + 1)

y = clean[LABEL]

cv = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

for name, X in [("log ratios", X_ratio), ("log raw gases", X_raw)]:
    for depth in [3, 5, None]:

        tree = DecisionTreeClassifier(
            max_depth=depth,
            random_state=42
        )

        scores = cross_val_score(
            tree,
            X,
            y,
            cv=cv,
            scoring="f1_macro"
        )

        print(
            f"{name:14s} depth={str(depth):4s} "
            f"macro-F1 = {scores.mean():.3f}"
        )

log ratios     depth=3    macro-F1 = 0.456
log ratios     depth=5    macro-F1 = 0.699
log ratios     depth=None macro-F1 = 0.741
log raw gases  depth=3    macro-F1 = 0.516
log raw gases  depth=5    macro-F1 = 0.640
log raw gases  depth=None macro-F1 = 0.731
